# Planetary Motion — Core Question 4
We assume the Sun is fixed at the origin and the planet starts at $\mathbf r_0=(R,0)$ with tangential velocity $\mathbf v_0=(0,v_0)$, where $v_0>0$. SI units are used unless stated otherwise. The velocity Verlet updates are adapted from Core 2.

Run Core 2 and Core 3 first. This section uses `next_r` and `next_v` from Core 2, and `total_energy` and the physical constants from Core 3. The energy function uses the default mass of 1 kg. The half-day time step is kept separate from Core 3.

In [ ]:
# Use constants from Core 3
q4_G = G
q4_M = Mass_sun
q4_R = R_orbit
q4_dt = 12 * 60 * 60
q4_v_circular = np.sqrt(q4_G * q4_M / q4_R)
q4_v_escape = np.sqrt(2 * q4_G * q4_M / q4_R)

# Simulate an orbit
def q4_simulate_orbit(speed, dt, steps):
    r = np.array([q4_R, 0.0])
    v = np.array([0.0, speed])
    positions = [r]
    for i in range(steps):
        r_next = next_r(r, v, dt)
        v = next_v(r, r_next, v, dt)
        r = r_next
        positions.append(r)
    return np.array(positions)

def q4_initial_energy(v):
    return total_energy(np.array([q4_R, 0.0]), np.array([0.0, v]))

# Period of a bound orbit
def q4_orbital_period(v):
    a = -q4_G * q4_M / (2 * q4_initial_energy(v))
    return 2 * np.pi * np.sqrt(a**3 / (q4_G * q4_M))

## Effect of initial speed
We compare five initial speeds below the escape speed. Each bound orbit is simulated for at least one theoretical orbital period, rather than using one year for every speed. This allows the more extended orbits to reach aphelion. The period follows from $T=2\pi\sqrt{a^3/(GM)}$, where $a=-GM/(2\mathcal E_0)$ and $\mathcal E_0=v_0^2/2-GM/R$.

In [ ]:
plt.style.use('default')
q4_speeds = [20, 25, 29.8, 35, 40]
q4_orbits = []
plt.figure(figsize=(9, 7))
for q4_speed in q4_speeds:
    q4_v = q4_speed * 1000
    q4_steps = int(q4_orbital_period(q4_v) / q4_dt) + 1
    q4_positions = q4_simulate_orbit(q4_v, q4_dt, q4_steps)
    q4_orbits.append(q4_positions)
    plt.plot(q4_positions[:, 0] / q4_R, q4_positions[:, 1] / q4_R,
             label=str(q4_speed) + " km/s")
plt.plot(0, 0, 'yo', label='Sun')
plt.plot(1, 0, 'ko', label='Start')
plt.axis('equal')
plt.xlabel('x (AU)')
plt.ylabel('y (AU)')
plt.title('Orbits for different initial speeds')
plt.legend()
plt.grid(True)
plt.show()

The orbit at 29.8 km/s is nearly circular. Lower speeds start near aphelion and move closer to the Sun, while higher speeds start at perihelion and reach larger distances. The 40 km/s orbit is much more extended than the other examples.

## Orbital parameters
Conservation of angular momentum gives $h=Rv_0$. At a turning point the radial velocity is zero, so
$$\mathcal E_0=\frac{h^2}{2r^2}-\frac{GM}{r}.$$
Substituting the initial energy gives a quadratic in $r$:
$$2\mathcal E_0r^2+2GMr-h^2=0.$$
One root is $R$ because the initial velocity is tangential. The other is
$$r_2=\frac{R^2v_0^2}{2GM-Rv_0^2}.$$
For $0<v_0<v_{\rm esc}$,
$$P=\min(R,r_2),\quad A=\max(R,r_2),\quad a=\frac{A+P}{2},\quad b=\sqrt{AP},\quad e=\frac{A-P}{A+P}.$$
We estimate $P$ and $A$ from the minimum and maximum simulated distance to the Sun, and calculate the other parameters using the same formulas.

In [ ]:
def q4_parameters(P, A):
    a = (A + P) / 2
    b = np.sqrt(A * P)
    e = (A - P) / (A + P)
    return P, A, a, b, e

# Theoretical ellipse parameters
def q4_ellipse_values(v):
    r2 = q4_R**2 * v**2 / (2 * q4_G * q4_M - q4_R * v**2)
    return q4_parameters(min(q4_R, r2), max(q4_R, r2))

# Compare theory and simulation
print('Speed | Source | P/R | A/R | a/R | b/R | e')
for q4_i in range(len(q4_speeds)):
    q4_speed = q4_speeds[q4_i]
    q4_theory = q4_ellipse_values(q4_speed * 1000)
    q4_positions = q4_orbits[q4_i]
    q4_distances = np.sqrt(q4_positions[:, 0]**2 + q4_positions[:, 1]**2)
    q4_numerical = q4_parameters(min(q4_distances), max(q4_distances))
    q4_P, q4_A, q4_a, q4_b, q4_e = q4_theory
    print(q4_speed, 'Theory', round(q4_P/q4_R, 5), round(q4_A/q4_R, 5),
          round(q4_a/q4_R, 5), round(q4_b/q4_R, 5), round(q4_e, 5))
    q4_P, q4_A, q4_a, q4_b, q4_e = q4_numerical
    print(q4_speed, 'Simulation', round(q4_P/q4_R, 5), round(q4_A/q4_R, 5),
          round(q4_a/q4_R, 5), round(q4_b/q4_R, 5), round(q4_e, 5))
    print('Maximum turning-point error (AU):',
          round(max(abs(q4_theory[0]-q4_numerical[0]), abs(q4_theory[1]-q4_numerical[1]))/q4_R, 6))
    print()

Small differences are expected because velocity Verlet has finite time-step error and the sampled positions may not fall exactly at the turning points. The absolute turning-point error is useful even for nearly circular orbits, where relative eccentricity error can be misleading.

## Classification of orbits
The initial specific orbital energy is
$$\mathcal E_0=\frac12v_0^2-\frac{GM}{R}.$$
For a nonzero tangential initial velocity, negative energy gives an ellipse (including a circle), zero energy gives a parabola and positive energy gives a hyperbola. We use a small absolute tolerance for floating-point round-off when comparing energy with zero. The three trajectories below are shown over the same finite time interval; an open trajectory does not by itself prove escape.

In [ ]:
def q4_orbit_type(v):
    E = q4_initial_energy(v)
    if abs(E) < 0.001:
        return 'Parabola'
    if E < 0:
        return 'Ellipse'
    return 'Hyperbola'

print('Speed (km/s) | Energy (J/kg) | Orbit')
for q4_v in [20000, 29800, 40000, q4_v_escape, 45000]:
    print(round(q4_v/1000, 6), round(q4_initial_energy(q4_v), 2), q4_orbit_type(q4_v))

# Compare three orbit types over one year
plt.figure(figsize=(8, 7))
for q4_v in [29800, q4_v_escape, 45000]:
    q4_positions = q4_simulate_orbit(q4_v, q4_dt, 730)
    plt.plot(q4_positions[:, 0]/q4_R, q4_positions[:, 1]/q4_R,
             label=q4_orbit_type(q4_v))
plt.plot(0, 0, 'yo', label='Sun')
plt.plot(1, 0, 'ko', label='Start')
plt.axis('equal')
plt.xlabel('x (AU)')
plt.ylabel('y (AU)')
plt.title('Elliptical, parabolic and hyperbolic trajectories')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Test the circular orbit and all three classifications
q4_P, q4_A, q4_a, q4_b, q4_e = q4_ellipse_values(q4_v_circular)
if abs(q4_P/q4_R-1) < 1e-8 and abs(q4_A/q4_R-1) < 1e-8 and q4_e < 1e-8:
    print('Circular orbit: PASS')
else:
    print('Circular orbit: FAIL')

q4_test_speeds = [20000, q4_v_escape, 45000]
q4_expected_types = ['Ellipse', 'Parabola', 'Hyperbola']
for q4_i in range(len(q4_test_speeds)):
    q4_result = q4_orbit_type(q4_test_speeds[q4_i])
    q4_expected = q4_expected_types[q4_i]
    if q4_result == q4_expected:
        print(q4_expected, ':', 'PASS')
    else:
        print(q4_expected, ':', 'FAIL')